# Final Model Training and Temporal Holdout Evaluation

The after-departure XGBoost configuration was selected using the 2019–2021 training period and the 2022 validation period.

In this notebook, the selected pipeline is retrained using all development data from 2019 through 2022. The frozen model is then scored against the January–August 2023 temporal holdout.

The 2023 holdout was not used for:

- model fitting
- hyperparameter tuning
- threshold selection
- model comparison

The target defines a delayed flight as an arrival delay of **15 minutes or more**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    TargetEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from xgboost import XGBClassifier

In [ ]:
from pathlib import Path


def find_project_root() -> Path:
    """
    Locate the project root whether the notebook is launched
    from the project root or from inside the notebooks folder.
    """
    current_path = Path.cwd().resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root could not be located. "
        "Run this notebook from inside the "
        "Flight_Delay_Prediction project folder."
    )


PROJECT_ROOT = find_project_root()

RAW_DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

RAW_DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
engineered_data_path = (
    PROCESSED_DATA_DIRECTORY
    / "engineered_flight_data.csv"
)

df = pd.read_csv(
    engineered_data_path,
    parse_dates=["FL_DATE"]
)

print("Loaded from:", engineered_data_path)
print("Dataset shape:", df.shape)
df.head()

## Final Feature Set

The final model predicts arrival delay immediately after departure.

Therefore, the feature set includes actual departure information:

- `DEP_TIME`
- `DEP_DELAY`

Arrival-related variables, post-arrival variables, and delay-cause variables remain excluded to prevent target leakage.

In [ ]:
selected_features = [
    "AIRLINE",
    "ORIGIN",
    "DEST",

    "CRS_DEP_TIME",
    "DEP_TIME",
    "DEP_DELAY",

    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DISTANCE",

    "YEAR",
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "QUARTER",

    "DEP_HOUR",
    "TIME_OF_DAY",
    "SEASON",
    "DISTANCE_CATEGORY",

    "IS_WEEKEND",
    "IS_PEAK_SEASON",
    "IS_BUSY_HOUR"
]

target = "IS_DELAYED"

print("Number of selected features:", len(selected_features))

## Final Chronological Split

The final development dataset combines observations from 2019 through 2022.

The January–August 2023 period is retained as the temporal holdout. Its labels are used only when the frozen final model is evaluated in this notebook.

In [ ]:
final_train_df = df[
    df["YEAR"] <= 2022
].copy()

test_df = df[
    df["YEAR"] == 2023
].copy()

In [ ]:
X_final_train = final_train_df[selected_features]
y_final_train = final_train_df[target]

X_test = test_df[selected_features]
y_test = test_df[target]

In [ ]:
final_split_summary = pd.DataFrame({
    "Dataset": [
        "Final Training",
        "Temporal Holdout Test"
    ],
    "Period": [
        "2019–2022",
        "January–August 2023"
    ],
    "Rows": [
        len(X_final_train),
        len(X_test)
    ]
})

display(final_split_summary)

final_training_target_summary = pd.DataFrame({
    "Count": (
        y_final_train
        .value_counts()
        .sort_index()
    ),
    "Percentage (%)": (
        y_final_train
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

final_training_target_summary.index = [
    "On Time (<15 min)",
    "Delayed (≥15 min)"
]

display(final_training_target_summary)

In [ ]:
numerical_features = [
    "CRS_DEP_TIME",
    "DEP_TIME",
    "DEP_DELAY",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DISTANCE",
    "YEAR",
    "MONTH",
    "DAY",
    "QUARTER",
    "DEP_HOUR",
    "IS_WEEKEND",
    "IS_PEAK_SEASON",
    "IS_BUSY_HOUR"
]

one_hot_features = [
    "DAY_OF_WEEK",
    "TIME_OF_DAY",
    "SEASON",
    "DISTANCE_CATEGORY"
]

target_encode_features = [
    "AIRLINE",
    "ORIGIN",
    "DEST"
]

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

one_hot_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

target_transformer = Pipeline(
    steps=[
        (
            "target_encoder",
            TargetEncoder(
                target_type="binary",
                random_state=42
            )
        )
    ]
)

In [ ]:
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        (
            "onehot",
            one_hot_transformer,
            one_hot_features
        ),
        (
            "target",
            target_transformer,
            target_encode_features
        )
    ],
    remainder="drop"
)

In [ ]:
final_xgb_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            final_preprocessor
        ),
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

final_xgb_pipeline

In [ ]:
print("Training the final XGBoost model...")

final_xgb_pipeline.fit(
    X_final_train,
    y_final_train
)

print("Final model training completed.")

In [ ]:
y_test_pred = final_xgb_pipeline.predict(
    X_test
)

y_test_prob = final_xgb_pipeline.predict_proba(
    X_test
)[:, 1]

In [ ]:
final_test_results = {
    "Model": "Final After-Departure XGBoost",
    "Accuracy": accuracy_score(
        y_test,
        y_test_pred
    ),
    "Precision": precision_score(
        y_test,
        y_test_pred,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        y_test_pred,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        y_test_pred,
        zero_division=0
    ),
    "ROC_AUC": roc_auc_score(
        y_test,
        y_test_prob
    )
}

pd.DataFrame(
    [final_test_results]
)

In [ ]:
print("Final Test Classification Report\n")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=[
            "On Time",
            "Delayed"
        ],
        zero_division=0
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    display_labels=[
        "On Time",
        "Delayed"
    ]
)

plt.title(
    "Final XGBoost Test Confusion Matrix"
)

plt.show()

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_test_prob
)

plt.title(
    "Final XGBoost ROC Curve"
)

plt.show()

In [ ]:
models_directory = MODELS_DIRECTORY

models_directory.mkdir(
    parents=True,
    exist_ok=True
)

print("Models directory:")
print(models_directory)

In [ ]:
model_path = (
    models_directory
    / "final_after_departure_xgboost_pipeline.joblib"
)

joblib.dump(
    final_xgb_pipeline,
    model_path
)

print(
    f"Final model saved to: {models_directory}"
)

In [ ]:
results_path = (
    models_directory
    / "final_test_metrics.csv"
)

pd.DataFrame(
    [final_test_results]
).to_csv(
    results_path,
    index=False
)

print(
    f"Final metrics saved to: {results_path}"
)


## Final Temporal Holdout Results

The final model metrics, classification report, confusion matrix, ROC curve, and precision-recall curve are displayed above.

These results measure how the frozen after-departure pipeline performs on the January–August 2023 temporal holdout.

The model relies on actual departure information, particularly `DEP_DELAY`. Therefore, it must be interpreted as an after-departure arrival-delay model and not as a pre-departure warning system.

The output from this notebook is the authoritative source for the final test metrics used by the README and Streamlit application.

In [ ]:
validation_test_comparison = pd.DataFrame([
    {
        "Dataset": "Validation",
        "Period": "2022",
        "Accuracy": 0.928082,
        "Precision": 0.920690,
        "Recall": 0.720150,
        "F1": 0.808165,
        "ROC_AUC": 0.933528
    },
    {
        "Dataset": "Temporal Holdout Test",
        "Period": "January–August 2023",
        "Accuracy": final_test_results["Accuracy"],
        "Precision": final_test_results["Precision"],
        "Recall": final_test_results["Recall"],
        "F1": final_test_results["F1"],
        "ROC_AUC": final_test_results["ROC_AUC"]
    }
])

validation_test_comparison

## Final Conclusion

The project first evaluated models using scheduled-flight, airport, airline, calendar, and seasonal information available before departure.

The corrected pre-departure experiments showed limited predictive performance. Temporal hyperparameter tuning produced only a negligible F1 improvement, indicating that parameter optimization alone could not overcome the limited signal in the available pre-departure variables.

The operational scenario was then changed to predict arrival delay immediately after departure. At this point, `DEP_TIME` and `DEP_DELAY` become available.

The after-departure XGBoost pipeline produced substantially stronger validation performance and was selected as the final configuration. It was retrained using data from 2019 through 2022 and scored against the January–August 2023 temporal holdout.

The final system must be described specifically as an **after-departure arrival-delay prediction model**. Its main limitation is that it cannot provide a prediction before the aircraft has departed.

## Save Deployment Metadata

The Streamlit application requires valid airline and airport options. These values are extracted from the final training dataset and saved separately so the application does not need to load the complete flight dataset.

In [ ]:
import hashlib
import platform

from importlib.metadata import version


model_sha256 = hashlib.sha256(
    model_path.read_bytes()
).hexdigest()

deployment_metadata = {
    "artifact_version": "2.0",
    "model_file": model_path.name,
    "model_sha256": model_sha256,

    "prediction_moment": (
        "immediately_after_departure"
    ),

    "target_definition": (
        "IS_DELAYED = 1 when ARR_DELAY >= 15 minutes"
    ),

    "class_labels": {
        0: "On Time (<15 min)",
        1: "Delayed (>=15 min)"
    },

    "decision_threshold": 0.5,
    "probability_calibrated": False,

    "training_period": (
        "2019-01-01 through 2022-12-31"
    ),

    "test_period": (
        "2023-01-01 through 2023-08-31"
    ),

    "training_scope": (
        "Completed, non-diverted flights only"
    ),

    "feature_names": selected_features,

    "package_versions": {
        "python": platform.python_version(),
        "pandas": version("pandas"),
        "numpy": version("numpy"),
        "scikit-learn": version("scikit-learn"),
        "xgboost": version("xgboost"),
        "joblib": version("joblib")
    },

    "validation_metrics": {
        "Accuracy": 0.928082,
        "Precision": 0.920690,
        "Recall": 0.720150,
        "F1": 0.808165,
        "ROC_AUC": 0.933528
    },

    "test_metrics": {
        key: value
        for key, value in final_test_results.items()
        if key != "Model"
    },

    "airlines": sorted(
        X_final_train["AIRLINE"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    ),

    "origins": sorted(
        X_final_train["ORIGIN"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    ),

    "destinations": sorted(
        X_final_train["DEST"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    ),

    "defaults": {
        "CRS_ELAPSED_TIME": float(
            X_final_train[
                "CRS_ELAPSED_TIME"
            ].median()
        ),

        "DISTANCE": float(
            X_final_train[
                "DISTANCE"
            ].median()
        )
    }
}

In [ ]:
metadata_path = (
    models_directory
    / "deployment_metadata.joblib"
)

joblib.dump(
    deployment_metadata,
    metadata_path
)

print(f"Deployment metadata saved to: {metadata_path}")

In [ ]:
print("Artifact version:")
print(deployment_metadata["artifact_version"])

print("\nPrediction moment:")
print(deployment_metadata["prediction_moment"])

print("\nTarget definition:")
print(deployment_metadata["target_definition"])

print("\nFeature count:")
print(len(deployment_metadata["feature_names"]))

print("\nAirlines:")
print(len(deployment_metadata["airlines"]))

print("\nOrigin airports:")
print(len(deployment_metadata["origins"]))

print("\nDestination airports:")
print(len(deployment_metadata["destinations"]))

print("\nModel SHA-256:")
print(deployment_metadata["model_sha256"])

print("\nPackage versions:")
print(deployment_metadata["package_versions"])